In [ ]:
import pandas as pd
import pywhatkit as kit
import random
import time
import logging
import os
from datetime import datetime

# =====================================================================
# 1. CONFIGURACIÓN DE RUTAS (Modifica estas rutas a tu gusto)
# =====================================================================
RUTA_EXCEL = r"C:\Usuario\Número_Usuario\Nube\...\Carpeta\DANIEL\Daniel_Excel\Cumpleaños\Fechas_Cumpleaños_Lista_Completa.xlsx"

# ---> AQUÍ ELIGES LA CARPETA ESPECÍFICA PARA TUS LOGS <---
CARPETA_LOGS = r"C:\Usuario\Número_Usuario\Nube\...\Carpeta\DANIEL\Daniel_Excel\Cumpleaños\Logs"
NOMBRE_ARCHIVO_LOG = "seguimiento_cumpleaños_.log"

# SEGURIDAD: Si la carpeta que elegiste no existe, Python la crea en el acto
if not os.path.exists(CARPETA_LOGS):
    os.makedirs(CARPETA_LOGS)

# Combinamos la carpeta y el nombre del archivo en una ruta válida para Windows
LOG_FILE = os.path.join(CARPETA_LOGS, NOMBRE_ARCHIVO_LOG)

# =====================================================================
# 2. CONFIGURACIÓN DEL SISTEMA DE LOGGING
# =====================================================================
logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    encoding='utf-8'
)

# --- 3. CANCIONES DE YOUTUBE PARA EL FESTEJADO ---
PLAYLIST_YOUTUBE = [
    'https://www.youtube.com/watch?v=inS9gAgSENE', # Stevie Wonder - Happy Birthday
    'https://www.youtube.com/watch?v=jqYxyd1iSNk', # Katy Perry - Birthday
    'https://youtu.be/MP1G8wnLpSM?si=l6pmEGRIz_t4JJae',
    'https://youtu.be/YJ84U_30GRM?si=iUeajQ3yLYGdLGwh',
    'https://youtu.be/40AJMQob25o?si=SEGejc3TK2HGvQDW',
    'https://youtu.be/l_NcVpR6DJo?si=N4lu8C3BQJLjh4oV',
    'https://youtu.be/YzR57xNtTHU?si=uC4-fJlq0fTdTIrN',
    'https://youtu.be/zVrVmDZyJYw?si=8SA5W1ElqCR2KFYU',
    'https://youtu.be/wpcgRcyB2ak?si=9wyVrotA3ogTEJgd',
    'https://youtu.be/IEHlJeQpn2M?si=7EPM1nzDlHxB910M'  
]

# --- 4. MENSAJES DIRECTOS PARA EL CUMPLEAÑERO ---
MENSAJES_CUMPLEAÑERO = [
    "¡Hola, {nombre}! ¡Feliz cumpleaños! 🎂 Hoy queremos recordarte lo valioso que eres para todo nuestro equipo de trabajo; tu dedicación diaria y compromiso hacen que alcancemos cada meta juntos. Como dato curioso, la paradoja del cumpleaños dice que en un grupo de 22 personas hay casi un 50% de probabilidad de que dos compartan fecha, ¡pero hoy el protagonismo es 100% tuyo! Te dedicamos esta canción para celebrar tu día: {link_video} ¡Un gran abrazo de parte de todos!",
    "¡Muchas felicidades en tu día, {nombre}! 🎈 Queríamos aprovechar hoy para agradecerte el esfuerzo y la excelente energía que aportas al equipo de trabajo; eres una pieza clave para sacar adelante cada jornada. Estadísticamente, la paradoja del cumpleaños nos dice que somos un grupo muy especial con alta probabilidad de coincidencias, ¡pero hoy no compartimos fecha, hoy te celebramos solo a ti! Te dejamos esta canción para festejar: {link_video} ¡Pásalo genial!",
    "¡Feliz día, {nombre}! 🍰 En las jornadas de trabajo pesado, contar con tu apoyo, talento y compañerismo es un verdadero privilegio para el equipo. Que tengas un día excelente y lleno de alegrías. Por cierto, la paradoja del cumpleaños dice que con nosotros 22 la probabilidad de coincidir es del 47.5%, así que es matemáticamente genial celebrar tu día hoy. Disfruta de esta canción que elegimos para ti: {link_video} ¡Felicidades!",
    "¡Hola, {nombre}! ¡Feliz vuelta al sol! 🎁 Tu constancia y buena actitud en el día a día son una gran inspiración para todos en el equipo de trabajo. Aunque la paradoja del cumpleaños sugiere que estadísticamente deberíamos cruzarnos en fechas en un grupo de nuestro tamaño, hoy el calendario se detiene exclusivamente para festejar tu vida. Te enviamos esta canción con mucho aprecio: {link_video} ¡Que tengas un día extraordinario!"
]

# --- 5. FUNCIÓN DE EJECUCIÓN ---
def ejecutar_bot_individual():
    logging.info("--- Iniciando comprobación diaria de cumpleaños ---")
    
    if not os.path.exists(RUTA_EXCEL):
        logging.error(f"No se encontró el archivo Excel en {RUTA_EXCEL}")
        return

    # Lectura del Excel
    df = pd.read_excel(RUTA_EXCEL)
    
    # CONVERSIÓN DE FECHAS A PRUEBA DE ERRORES
    df['FECHA_CUMPLEAÑOS'] = pd.to_datetime(
        df['FECHA_CUMPLEAÑOS'], 
        dayfirst=True, 
        errors='coerce'
    )
    
    # Auditoría interna de fechas dañadas
    filas_con_error = df[df['FECHA_CUMPLEAÑOS'].isna()]
    if not filas_con_error.empty:
        for _, fila in filas_con_error.iterrows():
            nombre_error = fila['NOMBRE'] if pd.notna(fila['NOMBRE']) else "Desconocido"
            logging.warning(f"Revisar Excel: La fecha de '{nombre_error}' no tiene un formato válido y fue omitida.")
    
    # Limpieza: Descartar las filas con fechas inválidas
    df = df.dropna(subset=['FECHA_CUMPLEAÑOS'])
    
    hoy = datetime.now().date()
    
    # BUSCAR SI ALGUIEN CUMPLE AÑOS HOY
    cumple_hoy = df[(df['FECHA_CUMPLEAÑOS'].dt.day == hoy.day) & 
                    (df['FECHA_CUMPLEAÑOS'].dt.month == hoy.month)]

    if not cumple_hoy.empty:
        for _, row in cumple_hoy.iterrows():
            nombre_festejado = row['NOMBRE']
            numero = str(row['CELULAR'])
            if not numero.startswith('+'): 
                numero = '+' + numero

            logging.info(f"Cumpleaños detectado HOY para: {nombre_festejado}")
            
            cancion_elegida = random.choice(PLAYLIST_YOUTUBE)
            mensaje_base = random.choice(MENSAJES_CUMPLEAÑERO)
            
            mensaje_final = mensaje_base.format(
                nombre=nombre_festejado, 
                link_video=cancion_elegida
            )

            try:
                # 1. Enviamos el texto, pero cambiamos tab_close a False. 
                # Si dejamos que pywhatkit intente cerrar la pestaña, lo hará antes de que enviemos el mensaje.
                kit.sendwhatmsg_instantly(numero, mensaje_final, wait_time=15, tab_close=False)
                
                # 2. Pausa crucial: Esperamos 3 segundos para garantizar que pywhatkit 
                # haya terminado de escribir/pegar todo el texto en el cuadro de chat.
                time.sleep(3)
                
                # 3. Forzamos el "ENTER" simulando tu teclado físico
                import pyautogui
                pyautogui.press('enter')
                logging.info(f"Presión de ENTER forzada para: {nombre_festejado}")
                
                # 4. Esperamos 3 segundos más para que WhatsApp procese y mande el mensaje (y cargue el link de YT)
                time.sleep(3)
                
                # 5. Cerramos la pestaña del navegador de forma segura usando el atajo de Windows (Ctrl + W)
                pyautogui.hotkey('ctrl', 'w')
                
                logging.info(f"Mensaje de cumpleaños enviado con éxito directamente a {nombre_festejado}")
                
                # Pausa de seguridad antes de pasar al siguiente cumpleañero (si hubiera más de uno)
                time.sleep(5) 
                
            except Exception as e:
                logging.error(f"Error enviando el mensaje a {nombre_festejado}: {e}")
    else:
        logging.info("No hay ningún cumpleaños programado para el día de hoy.")

    logging.info("--- Comprobación diaria finalizada ---")

if __name__ == "__main__":
    try:
        ejecutar_bot_individual()
    except Exception as e:
        logging.critical(f"Error crítico en el sistema: {e}")